In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [3]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
train.shape

(891, 12)

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [6]:
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

In [7]:
train['Age'] = train.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

In [8]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64

In [9]:
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})

In [10]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,S


In [11]:
train = pd.get_dummies(train, columns=['Embarked'], drop_first=True)

In [12]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,False,True
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,False,False
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,False,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,False,True
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,False,True


In [14]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)

In [15]:
train[['SibSp', 'Parch', 'FamilySize', 'IsAlone']].head()

,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1


In [16]:
train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

In [17]:
train['Title'].value_counts()

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Mlle          2
Major         2
Col           2
Countess      1
Capt          1
Ms            1
Sir           1
Lady          1
Mme           1
Don           1
Jonkheer      1
Name: count, dtype: int64

In [19]:
# французские/синонимичные написания → к основным
train['Title'] = train['Title'].replace(['Mlle', 'Ms'], 'Miss')
train['Title'] = train['Title'].replace('Mme', 'Mrs')

# все редкие → в одну группу Rare
rare = ['Dr', 'Rev', 'Major', 'Col', 'Countess', 'Capt', 'Sir', 'Lady', 'Don', 'Jonkheer']
train['Title'] = train['Title'].replace(rare, 'Rare')

In [20]:
train['Title'].value_counts()

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

In [21]:
train = pd.get_dummies(train, columns=['Title'], drop_first=True)

In [22]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_Q,Embarked_S,FamilySize,IsAlone,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,False,True,2,0,False,True,False,False
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,False,False,2,0,False,False,True,False
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,False,True,1,1,True,False,False,False
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,False,True,2,0,False,False,True,False
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,False,True,1,1,False,True,False,False


In [23]:
#try
train = pd.get_dummies(train, columns=['Pclass'], drop_first=True)

In [24]:
train = train.drop(columns=['Name', 'Ticket', 'Cabin', 'PassengerId'])

In [27]:
train = train.drop(columns=['SibSp', 'Parch'])

In [28]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Survived    891 non-null    int64  
 1   Sex         891 non-null    int64  
 2   Age         891 non-null    float64
 3   Fare        891 non-null    float64
 4   Embarked_Q  891 non-null    bool   
 5   Embarked_S  891 non-null    bool   
 6   FamilySize  891 non-null    int64  
 7   IsAlone     891 non-null    int64  
 8   Title_Miss  891 non-null    bool   
 9   Title_Mr    891 non-null    bool   
 10  Title_Mrs   891 non-null    bool   
 11  Title_Rare  891 non-null    bool   
 12  Pclass_2    891 non-null    bool   
 13  Pclass_3    891 non-null    bool   
dtypes: bool(8), float64(2), int64(4)
memory usage: 48.9 KB


In [29]:
train.head()

,Survived,Sex,Age,Fare,Embarked_Q,Embarked_S,FamilySize,IsAlone,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Pclass_2,Pclass_3
0,0,0,22.0,7.2500,False,True,2,0,False,True,False,False,False,True
1,1,1,38.0,71.2833,False,False,2,0,False,False,True,False,False,False
2,1,1,26.0,7.9250,False,True,1,1,True,False,False,False,False,True
3,1,1,35.0,53.1000,False,True,2,0,False,False,True,False,False,False
4,0,0,35.0,8.0500,False,True,1,1,False,True,False,False,False,True


In [31]:
# 1. загрузка
test = pd.read_csv('test.csv')

# 2. сохраняем PassengerId для сабмишена (ДО всех преобразований)
test_ids = test['PassengerId']

# 3. пропуски — пока Sex текст, а Pclass числа
test['Age'] = test.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))
test['Fare'] = test['Fare'].fillna(test['Fare'].median())

# 4. новые признаки — пока SibSp/Parch ещё на месте
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

# 5. титул
test['Title'] = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Title'] = test['Title'].replace(['Mlle', 'Ms'], 'Miss')
test['Title'] = test['Title'].replace('Mme', 'Mrs')
main_titles = ['Mr', 'Miss', 'Mrs', 'Master']
test['Title'] = test['Title'].apply(lambda t: t if t in main_titles else 'Rare')

# 6. кодируем Sex
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

# 7. one-hot — те же столбцы, что в train
test = pd.get_dummies(test, columns=['Embarked', 'Pclass', 'Title'], drop_first=True)

# 8. выравниваем столбцы ровно под train
feature_cols = train.drop(columns=['Survived']).columns
test = test.reindex(columns=feature_cols, fill_value=0)

In [32]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Sex         418 non-null    int64  
 1   Age         418 non-null    float64
 2   Fare        418 non-null    float64
 3   Embarked_Q  418 non-null    bool   
 4   Embarked_S  418 non-null    bool   
 5   FamilySize  418 non-null    int64  
 6   IsAlone     418 non-null    int64  
 7   Title_Miss  418 non-null    bool   
 8   Title_Mr    418 non-null    bool   
 9   Title_Mrs   418 non-null    bool   
 10  Title_Rare  418 non-null    bool   
 11  Pclass_2    418 non-null    bool   
 12  Pclass_3    418 non-null    bool   
dtypes: bool(8), float64(2), int64(3)
memory usage: 19.7 KB


In [33]:
from sklearn.linear_model import LinearRegression

# делим train на признаки (X) и ответ (y)
X = train.drop(columns=['Survived'])
y = train['Survived']

# создаём и обучаем модель
model = LinearRegression()
model.fit(X, y)

# применяем к test — получаем ЧИСЛА
raw_predictions = model.predict(test)

# смотрим на первые 10
raw_predictions[:10]

array([0.09134888, 0.60597126, 0.15521029, 0.07964239, 0.62630566,
       0.12650797, 0.63053014, 0.16813352, 0.77924408, 0.02218639])

In [34]:
# порог 0.5: оценка >= 0.5 → 1, иначе 0
predictions = (raw_predictions >= 0.5).astype(int)

# файл-сабмишен: ID из test + наши предсказания
submission = pd.DataFrame({'PassengerId': test_ids, 'Survived': predictions})

# сохраняем (index=False обязательно для Kaggle)
submission.to_csv('submission.csv', index=False)

submission.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [35]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# делим train: 80% учим, 20% проверяем
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

m = LinearRegression()
m.fit(X_tr, y_tr)
val_raw = m.predict(X_val)   # числа на проверочной части

# пробуем разные пороги
for t in [0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    acc = accuracy_score(y_val, (val_raw >= t).astype(int))
    print(f"порог {t}: точность {acc:.4f}")

порог 0.35: точность 0.8101
порог 0.4: точность 0.8156
порог 0.45: точность 0.8101
порог 0.5: точность 0.8268
порог 0.55: точность 0.8268
порог 0.6: точность 0.8101


In [36]:
# порог 0.55
predictions = (raw_predictions >= 0.55).astype(int)
submission = pd.DataFrame({'PassengerId': test_ids, 'Survived': predictions})
submission.to_csv('submission0.55.csv', index=False)
submission.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [37]:
# флаг «маленькая семья» (2–4)
train['SmallFamily'] = ((train['FamilySize'] >= 2) & (train['FamilySize'] <= 4)).astype(int)
test['SmallFamily']  = ((test['FamilySize'] >= 2) & (test['FamilySize'] <= 4)).astype(int)

# пересобираем X/y и выравниваем test под train
X = train.drop(columns=['Survived'])
y = train['Survived']
test = test.reindex(columns=X.columns, fill_value=0)

# быстрая проверка на валидации (Kaggle не трогаем)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
m = LinearRegression().fit(X_tr, y_tr)
val_raw = m.predict(X_val)
for t in [0.4, 0.45, 0.5, 0.55, 0.6]:
    acc = accuracy_score(y_val, (val_raw >= t).astype(int))
    print(f"порог {t}: {acc:.4f}")

порог 0.4: 0.8212
порог 0.45: 0.8212
порог 0.5: 0.8268
порог 0.55: 0.8268
порог 0.6: 0.8212


In [38]:
# убираем SmallFamily — он не помог (errors='ignore' = не падать, если его уже нет)
train = train.drop(columns=['SmallFamily'], errors='ignore')
test  = test.drop(columns=['SmallFamily'], errors='ignore')

# новый признак: ребёнок (Age < 16)
train['IsChild'] = (train['Age'] < 16).astype(int)
test['IsChild']  = (test['Age'] < 16).astype(int)

# пересобираем X/y и выравниваем test
X = train.drop(columns=['Survived'])
y = train['Survived']
test = test.reindex(columns=X.columns, fill_value=0)

# проверка на валидации
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
m = LinearRegression().fit(X_tr, y_tr)
val_raw = m.predict(X_val)
for t in [0.4, 0.45, 0.5, 0.55, 0.6]:
    acc = accuracy_score(y_val, (val_raw >= t).astype(int))
    print(f"порог {t}: {acc:.4f}")

порог 0.4: 0.8156
порог 0.45: 0.8045
порог 0.5: 0.8268
порог 0.55: 0.8268
порог 0.6: 0.7989


In [39]:
import numpy as np

# убираем IsChild — не помог
train = train.drop(columns=['IsChild'], errors='ignore')
test  = test.drop(columns=['IsChild'], errors='ignore')

# log(Fare): заменяем перекошенный Fare сглаженным (с защитой от повторного запуска)
if 'Fare' in train.columns:
    train['FareLog'] = np.log1p(train['Fare'])
    train = train.drop(columns=['Fare'])
if 'Fare' in test.columns:
    test['FareLog'] = np.log1p(test['Fare'])
    test = test.drop(columns=['Fare'])

# пересобираем X/y и выравниваем test
X = train.drop(columns=['Survived'])
y = train['Survived']
test = test.reindex(columns=X.columns, fill_value=0)

# проверка на валидации
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
m = LinearRegression().fit(X_tr, y_tr)
val_raw = m.predict(X_val)
for t in [0.4, 0.45, 0.5, 0.55, 0.6]:
    acc = accuracy_score(y_val, (val_raw >= t).astype(int))
    print(f"порог {t}: {acc:.4f}")

порог 0.4: 0.8156
порог 0.45: 0.8101
порог 0.5: 0.8268
порог 0.55: 0.8268
порог 0.6: 0.8101


In [40]:
# взаимодействие пол × класс
train['Sex_x_P2'] = train['Sex'] * train['Pclass_2'].astype(int)
train['Sex_x_P3'] = train['Sex'] * train['Pclass_3'].astype(int)
test['Sex_x_P2']  = test['Sex']  * test['Pclass_2'].astype(int)
test['Sex_x_P3']  = test['Sex']  * test['Pclass_3'].astype(int)

# пересобираем X/y и выравниваем test
X = train.drop(columns=['Survived'])
y = train['Survived']
test = test.reindex(columns=X.columns, fill_value=0)

# проверка на валидации
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
m = LinearRegression().fit(X_tr, y_tr)
val_raw = m.predict(X_val)
for t in [0.4, 0.45, 0.5, 0.55, 0.6]:
    acc = accuracy_score(y_val, (val_raw >= t).astype(int))
    print(f"порог {t}: {acc:.4f}")

порог 0.4: 0.8212
порог 0.45: 0.8212
порог 0.5: 0.8212
порог 0.55: 0.8268
порог 0.6: 0.7821
